In [1]:
import os
import h5py
import pandas as pd
import numpy as np

In [2]:
def collect_probabilities_df(h5_path, patch_size=(224,224), stride_factor=0.5):
    """
    Collects and averages probabilities and attention scores from sliding windows across subregions
    using a pandas DataFrame for more efficient manipulation.
    
    sliding_windows: List of (top_left_coords, probability, attention_score) for each window
    patch_size: Size of each patch (tuple: (height, width))
    stride_factor: How much each window moves, 0.5 means half patch overlap
    
    Returns:
    - avg_df: DataFrame with averaged probabilities and attention scores for each subregion
    """
    h5data = h5py.File(h5_path, 'r')
    
    stride_h = int(patch_size[0] * stride_factor)  # Vertical stride
    stride_w = int(patch_size[1] * stride_factor)  # Horizontal stride
    
    coords = h5data['coords'][:]
    attention_scores =h5data['attention_scores'][:]
    weighted_tile_probabilities = h5data['weighted_tile_probabilities'][:]
    tile_probabilities = h5data['tile_probabilities'][:]
    
    
    # Prepare a list to collect data for the DataFrame
    data = []

    # for (top_left, prob, attn_score) in h5data:
    for idx in range(len(coords)):
        x, y = coords[idx]
        
        # Define 4 subregions (assuming 1/4 division of patch)
        subregions = [
            (x, y),  # Top-left
            (x + stride_w, y),  # Top-right
            (x, y + stride_h),  # Bottom-left
            (x + stride_w, y + stride_h),  # Bottom-right
        ]
        
        for region in subregions:
            data.append({
                'coords': region,
                'weighted_tile_probabilities': weighted_tile_probabilities[idx],
                'tile_probabilities': tile_probabilities[idx],
                'attention_scores': attention_scores[idx]
            })

    # Convert data to a pandas DataFrame
    df = pd.DataFrame(data)
    
    # Group by 'region' and compute mean for each region
    # avg_df = df.groupby('coords').mean().reset_index()

    avg_df = df.groupby('coords').agg(
        avg_probability=('tile_probabilities', 'mean'),
        avg_attention_score=('attention_scores', 'mean'),
        avg_weighted_tile_prob=('weighted_tile_probabilities', 'mean'),
        tile_probabilities=('tile_probabilities', lambda x: list(x)),
        attention_scores=('attention_scores', lambda x: list(x)),
        weighted_tile_probabilities=('weighted_tile_probabilities', lambda x: list(x))
    ).reset_index()
    
    return avg_df


In [3]:
def save_hdf5(output_path, asset_dict, attr_dict= None, mode='w'):
    file = h5py.File(output_path, mode)
    for key, val in asset_dict.items():
        data_shape = val.shape
        if key not in file:
            data_type = val.dtype
            chunk_shape = (1, ) + data_shape[1:]
            maxshape = (None, ) + data_shape[1:]
            dset = file.create_dataset(key, shape=data_shape, maxshape=maxshape, chunks=chunk_shape, dtype=data_type)
            dset[:] = val
            if attr_dict is not None:
                if key in attr_dict.keys():
                    for attr_key, attr_val in attr_dict[key].items():
                        dset.attrs[attr_key] = attr_val
        else:
            dset = file[key]
            dset.resize(len(dset) + data_shape[0], axis=0)
            dset[-data_shape[0]:] = val
    file.close()
    return output_path

In [4]:
HEATMAP_OUTPUT_path = "/rsrch5/home/trans_mol_path/cercan/data/BE/aneuploid/clam/trainings/sep24_progress/1_0/heatmap/16768515_21_s1_fold3_MDA_window05/raw/HEATMAP_OUTPUT/"

In [5]:
for flow_group in ['aneuploid', 'diploid']:
    for subdir, _, _ in os.walk(os.path.join(HEATMAP_OUTPUT_path,flow_group)):
        files = os.listdir(subdir)
        blockmap_path = os.path.join(subdir , os.path.basename(subdir)+'_blockmap.h5')
        if os.path.isfile(blockmap_path):
            new_file_path = blockmap_path[:-3]+'05window.h5'
            avg_df = collect_probabilities_df(blockmap_path)
            coords = np.array(avg_df['coords'].tolist(), dtype=int)
            attention_scores = np.array(avg_df['attention_scores'].tolist(), dtype=float)
            tile_probabilities = np.array(avg_df['tile_probabilities'].tolist(), dtype=float)
            weighted_tile_probabilities = np.array(avg_df['weighted_tile_probabilities'].tolist(), dtype=float)
            
            asset_dict = { 'coords': coords, 'attention_scores': attention_scores, 'tile_probabilities':tile_probabilities,'weighted_tile_probabilities':weighted_tile_probabilities}
            save_hdf5(new_file_path, asset_dict)



In [15]:
'/rsrch5/home/trans_mol_path/cercan/data/BE/aneuploid/clam/trainings/sep24_progress/1_0/heatmap/16768515_21_s1_fold3_NU_window05/raw/HEATMAP_OUTPUT/diploid/D1041-2020-11-17_13.42.54_ROI0/D1041-2020-11-17_13.42.54_ROI0_blockmap05window.h5'

'/rsrch5/home/trans_mol_path/cercan/data/BE/aneuploid/clam/trainings/sep24_progress/1_0/heatmap/16768515_21_s1_fold3_NU_window05/raw/HEATMAP_OUTPUT/diploid/D1041-2020-11-17_13.42.54_ROI0/D1041-2020-11-17_13.42.54_ROI0_blockmap05window.h5'

In [6]:
test_file = "/rsrch5/home/trans_mol_path/cercan/data/BE/aneuploid/clam/trainings/sep24_progress/1_0/heatmap/16768515_21_s1_fold3_NU_window05/raw/HEATMAP_OUTPUT/aneuploid/D1056-2020-11-17_11.31.01_ROI0/D1056-2020-11-17_11.31.01_ROI0_blockmap.h5"

In [7]:
d115_file = h5py.File(test_file, 'r')
d115_file.keys()


<KeysViewHDF5 ['attention_scores', 'coords', 'raw_attention_scores', 'tile_probabilities', 'weighted_tile_probabilities']>

In [8]:
d115_file['attention_scores']

<HDF5 dataset "attention_scores": shape (314, 1), type "<f4">

In [7]:
d115_file['coords']

<HDF5 dataset "coords": shape (314, 2), type "<i8">

In [8]:
d115_file['attention_scores']

<HDF5 dataset "attention_scores": shape (314, 1), type "<f4">

In [13]:
avg_df = collect_probabilities_df(test_file)
avg_df

,coords,avg_probability,avg_attention_score,avg_weighted_tile_prob,tile_probabilities,attention_scores,weighted_tile_probabilities
0,"(452, 2192)",0.989500,[0.0037040077],0.00,[0.9894999861717224],[[0.0037040077]],[0.0]
1,"(452, 2304)",0.990600,[0.0036402894],0.00,"[0.9894999861717224, 0.9916999936103821]","[[0.0037040077], [0.003576571]]","[0.0, 0.0]"
2,"(452, 2416)",0.993600,[0.0036389334],0.00,"[0.9916999936103821, 0.9955000281333923]","[[0.003576571], [0.0037012957]]","[0.0, 0.0]"
3,"(452, 2528)",0.995600,[0.003584764],0.00,"[0.9955000281333923, 0.9957000017166138]","[[0.0037012957], [0.003468232]]","[0.0, 0.0]"
4,"(452, 2640)",0.995850,[0.0034577884],0.00,"[0.9957000017166138, 0.9959999918937683]","[[0.003468232], [0.0034473445]]","[0.0, 0.0]"
...,...,...,...,...,...,...,...
366,"(2132, 3760)",0.757625,[0.0012091162],0.25,"[1.0, 1.0, 1.0, 0.030500000342726707]","[[0.0021137493], [0.0008234074], [0.0008136375...","[0.0, 0.0, 0.0, 1.0]"
367,"(2132, 3872)",0.515250,[0.00095453905],0.50,"[1.0, 0.030500000342726707]","[[0.0008234074], [0.0010856707]]","[0.0, 1.0]"
368,"(2244, 3648)",1.000000,[0.0008136375],0.00,[1.0],[[0.0008136375]],[0.0]
369,"(2244, 3760)",0.515250,[0.0009496541],0.50,"[1.0, 0.030500000342726707]","[[0.0008136375], [0.0010856707]]","[0.0, 1.0]"


In [13]:
coords = np.array(avg_df['coords'].tolist(), dtype=int)
attention_scores = np.array(avg_df['attention_scores'].tolist(), dtype=float)
tile_probabilities = np.array(avg_df['tile_probabilities'].tolist(), dtype=float)
weighted_tile_probabilities = np.array(avg_df['weighted_tile_probabilities'].tolist(), dtype=float)

In [14]:
weighted_tile_probabilities.shape

(371,)